# Dataset Analysis - Phase 2 IQA Training Data

**Purpose**: Exploratory Data Analysis (EDA) for Phase 2 Image Quality Assessment training datasets

**Datasets Analyzed**:
- IQA Phase 2 Training Data (`data/training/iqa_phase2/`)
- OHR-Bench Benchmark (`data/benchmarks/ohr-bench/`)

**Analysis Goals**:
1. Image resolution distribution
2. Quality distribution (OHR-Bench)
3. Class balance analysis
4. Dataset characteristics and statistics

In [ ]:
# Imports
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from collections import Counter
import json

# Setup plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Add project root to path
project_root = Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd()
sys.path.insert(0, str(project_root / 'src'))

print(f"Project root: {project_root}")
print(f"Python version: {sys.version}")

## 1. Dataset Paths and Overview

In [ ]:
# Dataset paths
iqa_training_path = project_root / "data" / "training" / "iqa_phase2"
ohr_bench_path = project_root / "data" / "benchmarks" / "ohr-bench"

# Check if datasets exist
print(f"IQA Training Data exists: {iqa_training_path.exists()}")
print(f"OHR-Bench exists: {ohr_bench_path.exists()}")

if iqa_training_path.exists():
    print(f"\nIQA Training subdirectories:")
    for subdir in sorted(iqa_training_path.iterdir()):
        if subdir.is_dir():
            file_count = len(list(subdir.rglob('*.*')))
            print(f"  {subdir.name}: {file_count} files")

## 2. Image Resolution Distribution

In [ ]:
def analyze_image_resolutions(dataset_path, max_samples=1000):
    """Analyze image resolutions in a dataset."""
    image_extensions = {'.png', '.jpg', '.jpeg', '.tiff', '.tif'}
    resolutions = []
    widths = []
    heights = []
    dpi_values = []
    
    # Sample images
    all_images = [p for p in dataset_path.rglob('*') if p.suffix.lower() in image_extensions]
    sample_images = all_images[:max_samples] if len(all_images) > max_samples else all_images
    
    print(f"Analyzing {len(sample_images)} images (total: {len(all_images)})...")
    
    for img_path in sample_images:
        try:
            with Image.open(img_path) as img:
                width, height = img.size
                resolutions.append((width, height))
                widths.append(width)
                heights.append(height)
                
                # Get DPI if available
                dpi = img.info.get('dpi', (None, None))
                if dpi[0]:
                    dpi_values.append(dpi[0])
        except Exception as e:
            print(f"Error processing {img_path.name}: {e}")
    
    return {
        'resolutions': resolutions,
        'widths': widths,
        'heights': heights,
        'dpi_values': dpi_values,
        'total_images': len(all_images),
        'sampled_images': len(sample_images)
    }

# Analyze IQA training data
if iqa_training_path.exists():
    iqa_stats = analyze_image_resolutions(iqa_training_path, max_samples=1000)
    
    # Plot distributions
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Width distribution
    axes[0, 0].hist(iqa_stats['widths'], bins=50, edgecolor='black')
    axes[0, 0].set_title('Image Width Distribution')
    axes[0, 0].set_xlabel('Width (pixels)')
    axes[0, 0].set_ylabel('Frequency')
    axes[0, 0].axvline(np.mean(iqa_stats['widths']), color='r', linestyle='--', label=f"Mean: {np.mean(iqa_stats['widths']):.0f}")
    axes[0, 0].legend()
    
    # Height distribution
    axes[0, 1].hist(iqa_stats['heights'], bins=50, edgecolor='black')
    axes[0, 1].set_title('Image Height Distribution')
    axes[0, 1].set_xlabel('Height (pixels)')
    axes[0, 1].set_ylabel('Frequency')
    axes[0, 1].axvline(np.mean(iqa_stats['heights']), color='r', linestyle='--', label=f"Mean: {np.mean(iqa_stats['heights']):.0f}")
    axes[0, 1].legend()
    
    # Resolution scatter
    widths_sample = iqa_stats['widths'][:500]
    heights_sample = iqa_stats['heights'][:500]
    axes[1, 0].scatter(widths_sample, heights_sample, alpha=0.5)
    axes[1, 0].set_title('Resolution Scatter Plot')
    axes[1, 0].set_xlabel('Width (pixels)')
    axes[1, 0].set_ylabel('Height (pixels)')
    
    # DPI distribution
    if iqa_stats['dpi_values']:
        axes[1, 1].hist(iqa_stats['dpi_values'], bins=20, edgecolor='black')
        axes[1, 1].set_title('DPI Distribution')
        axes[1, 1].set_xlabel('DPI')
        axes[1, 1].set_ylabel('Frequency')
        axes[1, 1].axvline(300, color='g', linestyle='--', label='Target: 300 DPI')
        axes[1, 1].legend()
    else:
        axes[1, 1].text(0.5, 0.5, 'No DPI information available', ha='center', va='center')
        axes[1, 1].set_title('DPI Distribution')
    
    plt.tight_layout()
    plt.show()
    
    # Print statistics
    print("\n=== Resolution Statistics ===")
    print(f"Total images: {iqa_stats['total_images']:,}")
    print(f"Sampled: {iqa_stats['sampled_images']:,}")
    print(f"\nWidth: {np.mean(iqa_stats['widths']):.0f} ± {np.std(iqa_stats['widths']):.0f} px")
    print(f"Height: {np.mean(iqa_stats['heights']):.0f} ± {np.std(iqa_stats['heights']):.0f} px")
    if iqa_stats['dpi_values']:
        print(f"DPI: {np.mean(iqa_stats['dpi_values']):.0f} ± {np.std(iqa_stats['dpi_values']):.0f}")


## 3. OHR-Bench Quality Distribution

In [ ]:
# Load OHR-Bench dataset
if ohr_bench_path.exists():
    ohr_bench_file = ohr_bench_path / "OHR-Bench_v2.parquet"
    
    if ohr_bench_file.exists():
        ohr_df = pd.read_parquet(ohr_bench_file)
        
        print(f"OHR-Bench Dataset Shape: {ohr_df.shape}")
        print(f"\nColumns: {list(ohr_df.columns)}")
        print(f"\nFirst few rows:")
        display(ohr_df.head())
        
        # Analyze quality distribution if quality column exists
        quality_cols = [col for col in ohr_df.columns if 'quality' in col.lower() or 'score' in col.lower()]
        
        if quality_cols:
            print(f"\nQuality-related columns: {quality_cols}")
            
            fig, axes = plt.subplots(1, len(quality_cols), figsize=(6*len(quality_cols), 5))
            if len(quality_cols) == 1:
                axes = [axes]
            
            for idx, col in enumerate(quality_cols):
                if pd.api.types.is_numeric_dtype(ohr_df[col]):
                    axes[idx].hist(ohr_df[col].dropna(), bins=50, edgecolor='black')
                    axes[idx].set_title(f'{col} Distribution')
                    axes[idx].set_xlabel(col)
                    axes[idx].set_ylabel('Frequency')
                    axes[idx].axvline(ohr_df[col].mean(), color='r', linestyle='--', 
                                     label=f"Mean: {ohr_df[col].mean():.2f}")
                    axes[idx].legend()
                else:
                    value_counts = ohr_df[col].value_counts()
                    value_counts.plot(kind='bar', ax=axes[idx])
                    axes[idx].set_title(f'{col} Distribution')
                    axes[idx].set_xlabel(col)
                    axes[idx].set_ylabel('Count')
            
            plt.tight_layout()
            plt.show()
    else:
        print(f"OHR-Bench parquet file not found at {ohr_bench_file}")
else:
    print("OHR-Bench directory not found")

## 4. Class Balance Analysis

In [ ]:
def analyze_class_balance(dataset_path):
    """Analyze class distribution from directory structure or labels."""
    
    # Check for labels directory
    labels_path = dataset_path / "labels"
    
    if labels_path.exists():
        print("Analyzing labels from labels directory...")
        label_files = list(labels_path.glob("*.json"))
        
        if label_files:
            all_labels = []
            for label_file in label_files[:1000]:  # Sample first 1000
                try:
                    with open(label_file) as f:
                        labels = json.load(f)
                        if isinstance(labels, dict) and 'labels' in labels:
                            all_labels.append(labels['labels'])
                except Exception as e:
                    print(f"Error reading {label_file.name}: {e}")
            
            if all_labels:
                # Aggregate label counts
                label_counter = Counter()
                for labels in all_labels:
                    if isinstance(labels, dict):
                        for key, value in labels.items():
                            if value.get('value', 0) == 1:
                                label_counter[key] += 1
                
                return label_counter
    
    # Fallback: analyze directory structure
    print("Analyzing class balance from directory structure...")
    subdirs = [d for d in dataset_path.iterdir() if d.is_dir()]
    
    class_counts = {}
    for subdir in subdirs:
        file_count = len(list(subdir.rglob('*.*')))
        class_counts[subdir.name] = file_count
    
    return class_counts

# Analyze IQA training data
if iqa_training_path.exists():
    class_distribution = analyze_class_balance(iqa_training_path)
    
    if class_distribution:
        # Plot class distribution
        plt.figure(figsize=(12, 6))
        classes = list(class_distribution.keys())
        counts = list(class_distribution.values())
        
        plt.bar(classes, counts, edgecolor='black')
        plt.title('Class Distribution')
        plt.xlabel('Class')
        plt.ylabel('Count')
        plt.xticks(rotation=45, ha='right')
        
        # Add mean line
        mean_count = np.mean(counts)
        plt.axhline(mean_count, color='r', linestyle='--', label=f'Mean: {mean_count:.0f}')
        plt.legend()
        
        plt.tight_layout()
        plt.show()
        
        # Print statistics
        print("\n=== Class Balance Statistics ===")
        total_samples = sum(counts)
        for class_name, count in sorted(class_distribution.items(), key=lambda x: x[1], reverse=True):
            percentage = (count / total_samples) * 100
            print(f"{class_name:30s}: {count:6d} ({percentage:5.2f}%)")
        
        # Calculate balance metrics
        max_count = max(counts)
        min_count = min(counts)
        imbalance_ratio = max_count / min_count if min_count > 0 else float('inf')
        
        print(f"\nTotal samples: {total_samples:,}")
        print(f"Number of classes: {len(classes)}")
        print(f"Imbalance ratio (max/min): {imbalance_ratio:.2f}")
        print(f"Mean samples per class: {mean_count:.0f}")
        print(f"Std deviation: {np.std(counts):.0f}")


## 5. Summary and Findings

### Key Observations:

1. **Resolution Distribution**:
   - Mean image dimensions
   - DPI consistency
   - Outliers and edge cases

2. **Quality Distribution** (OHR-Bench):
   - Quality score range
   - Distribution shape
   - Correlation with OCR difficulty

3. **Class Balance**:
   - Most/least represented classes
   - Imbalance ratio
   - Recommendations for balancing

### Recommendations:

- [ ] Document any data quality issues found
- [ ] Plan data augmentation strategy for underrepresented classes
- [ ] Set appropriate batch sizes based on image dimensions
- [ ] Consider stratified sampling for training/validation splits

---
*Analysis Date: 2025-11-16*
*Analyst: Claude Code*